# Amazon Sales Intelligence & Business Analytics

## 1. Project Overview
This notebook provides a comprehensive Exploratory Data Analysis (EDA) of the cleaned Amazon Sales dataset. The primary objective is to uncover actionable business insights regarding revenue drivers, product category performance, geographical trends, and order fulfillment metrics. 

The dataset used here has already undergone a rigorous cleaning pipeline to correct geographical anomalies and reclassify product categories.

## 2. Import Libraries
We begin by importing the necessary Python libraries for data manipulation and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## 3. Load Dataset
Load the processed dataset from the `data/processed` directory.

In [ ]:
df = pd.read_csv('../data/processed/Amazon_Cleaned.csv')

# Convert OrderDate back to datetime (as CSV loads it as string)
df['OrderDate'] = pd.to_datetime(df['OrderDate'])

df.head()

## 4. Dataset Shape and Data Types
Understanding the dimensions of the dataset and the data types of its columns.

In [ ]:
print(f"Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns.\n")
df.info()

## 5. Data Quality Checks
Verify that there are no missing values or duplicates remaining after the pipeline execution.

In [ ]:
missing_values = df.isnull().sum()
duplicates = df.duplicated().sum()

print("Missing values per column:\n", missing_values[missing_values > 0])
print(f"\nTotal duplicate rows: {duplicates}")

## 6. Descriptive Statistics
A statistical summary of the numerical features.

In [ ]:
df[['Quantity', 'UnitPrice', 'Discount', 'Tax', 'ShippingCost', 'TotalAmount', 'NetRevenue']].describe()

## 7. Sales and Revenue Analysis
Calculating high-level Key Performance Indicators (KPIs).

In [ ]:
total_orders = df['OrderID'].nunique()
total_units = df['Quantity'].sum()
gross_revenue = df['TotalAmount'].sum()
aov = gross_revenue / total_orders if total_orders > 0 else 0

print(f"Total Orders: {total_orders:,}")
print(f"Total Units Sold: {total_units:,}")
print(f"Gross Revenue: ${gross_revenue:,.2f}")
print(f"Average Order Value (AOV): ${aov:,.2f}")

## 8. Category Analysis
Analyzing which product categories generate the most revenue and volume.

In [ ]:
category_performance = df.groupby('Category').agg(
    Total_Orders=('OrderID', 'nunique'),
    Gross_Revenue=('TotalAmount', 'sum')
).reset_index().sort_values(by='Gross_Revenue', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=category_performance, x='Gross_Revenue', y='Category', palette='viridis')
plt.title('Gross Revenue by Product Category')
plt.xlabel('Gross Revenue ($)')
plt.ylabel('Category')
plt.show()

## 9. State/Location Analysis
Identifying the top performing geographical regions.

In [ ]:
state_revenue = df.groupby('State')['TotalAmount'].sum().reset_index().sort_values('TotalAmount', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=state_revenue, x='State', y='TotalAmount', palette='magma')
plt.title('Top 10 States by Revenue')
plt.xlabel('State')
plt.ylabel('Gross Revenue ($)')
plt.show()

## 10. Order Status Analysis
Evaluating fulfillment success and defect rates (Returns/Cancellations).

In [ ]:
status_counts = df['OrderStatus'].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(status_counts, labels=status_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Order Status Distribution')
plt.show()

## 11. Top Products Analysis
Highlighting the best-selling products by Net Revenue.

In [ ]:
top_products = df.groupby('ProductName')['NetRevenue'].sum().reset_index().sort_values('NetRevenue', ascending=False).head(10)
top_products.style.format({'NetRevenue': '${:,.2f}'}).background_gradient(cmap='Greens')

## 12. Monthly Revenue Trend
Tracking sales performance over time to identify seasonality and growth.

In [ ]:
monthly_trend = df.groupby('OrderYearMonth')['TotalAmount'].sum().reset_index()

plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_trend, x='OrderYearMonth', y='TotalAmount', marker='o', linewidth=2.5)
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Gross Revenue ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. Key Business Insights
Based on the analysis above, we have derived the following insights:
1. **Category Dominance:** Specific categories consistently drive the majority of the top-line revenue, which should be the primary focus for inventory and marketing spend.
2. **Geographical Concentration:** A significant portion of revenue is concentrated in top states (e.g., TX). Localized marketing campaigns in these states will likely yield a high ROI.
3. **Fulfillment Health:** The order status distribution highlights the percentage of returned and cancelled orders. A high defect rate in specific categories might indicate product quality issues or inaccurate descriptions.
4. **Seasonality:** The monthly revenue trend line uncovers peak purchasing periods, essential for forecasting and supply chain management.

## 14. Conclusion
This exploratory analysis validates the structural integrity of the cleaned dataset and confirms its readiness for advanced dashboarding and SQL-based intelligence. The metrics derived here align perfectly with the established Key Performance Indicators (KPIs) required by the business stakeholders.